In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp Newton cho Hệ phi tuyến

$X_{k+1} = X_k - {J(X_k)}^{-1} F(X_k)$, với $J(X) = \partial F(X) = \dfrac{\partial(f_1, f_2, \cdots)}{\partial(x_1, x_2, \cdots)}$

# Thuật toán

#### Input:
- Hàm $F(X)=(f_1, f_2,\ldots,f_n)$
- Sai số $\varepsilon$
#### Output: nghiệm gần đúng X

#### Thuật toán:
- Bước 1: Tính $J(X)=[a_{ij}]_{n \times n}$ với $a_{ij}=\dfrac{\partial f_i}{\partial x_j}$
- Bước 2: Chọn điểm bắt đầu $X_0$, khởi tạo $k=0$. Lặp đến khi thoả mãn điều kiện dừng:
    - Tính $X_{k+1}=X_{k}-J(X_k)^{-1}F(X_k)$
    - Nếu thoả mãn điều kiện dừng: dừng chương trình. Trái lại cập nhật $k=k+1$
- Bước 3: Sau khi thoả mãn điều kiện dừng với $n$ lần lặp, thu được nghiệm gần đúng. Dừng chương trình

In [3]:
def newton_method(initial_values, tol=1e-6, max_iter=100, *funcs):
    n = len(initial_values)  # Number of initial values (and functions)
    x = np.array(initial_values, dtype=float)  # Convert initial values to a NumPy array
    results = []  
    
    for i in range(max_iter):
        F = np.array([func(*x) for func in funcs[:n]])
        
        # Create the Jacobian matrix using pandas
        J = pd.DataFrame(funcs[n](*x))
        
        if i == 0:
            print("J(X_0) =")
            print(J.to_string(index=False, header=False))
            print("\nJ(X_0)^-1 =")
            J_inv = pd.DataFrame(np.linalg.inv(J.values))
            print(J_inv.to_string(index=False, header=False))
            print("-" * 50)
        
        delta_x = np.linalg.solve(J.values, -F)
        x_new = x + delta_x
        
        # Calculate the L1 norm (sum of absolute differences)
        norm_diff = np.sum(np.abs(x_new - x))
        results.append([i] + list(x_new) + [norm_diff])
        x = x_new
        
        if norm_diff < tol:
            break
    
    columns = ['k'] + [f'x_({j+1},k+1)' for j in range(n)] + ['norm_diff']
    df = pd.DataFrame(results, columns=columns)
    print(df.to_string(index=False))
    
    return x

Bài ví dụ trong slide

Câu 21. Dùng phương pháp lặp Newton tính đến xấp xỉ nghiệm thứ năm của các phương trình dưới đây với xấp xỉ ban đầu là $X_0=(0,0)^{T}$

Câu 21a:

Câu 21b:

Câu 21c:

In [5]:
def f1(x, y):
    return x**2 + 2*y**2 - 3

def f2(x, y):
    return 5*y**2 + x*y - 4

# Jacobian matrix function
def jacobian(x, y):
    return [
        [2*x, 4*y],
        [y, 10*y+x]
    ]

# Store functions and Jacobian function
funcs = [f1, f2, jacobian]

# Initial guess
initial_values = [-1, -1]

# Perform Newton-Raphson method
solution = newton_method(initial_values, 1e-10, 5, *funcs)
print("Approximate solution:", solution)

J(X_0) =
-2.0  -4.0
-1.0 -11.0

J(X_0)^-1 =
-0.611111111111111  0.222222222222222
 0.055555555555556 -0.111111111111111
--------------------------------------------------
 k          x_(1,k+1)          x_(2,k+1)         norm_diff
 0 -1.444444444444444 -0.777777777777778 0.666666666666667
 1 -1.350662589194699 -0.769622833843017 0.101936799184506
 2 -1.347155774966865 -0.769799869577398 0.003683849962215
 3 -1.347150628120400 -0.769800358918396 0.000005636187463
 4 -1.347150628109127 -0.769800358919501 0.000000000012378
Approximate solution: [-1.34715063 -0.76980036]


# Phương pháp Newton modified
$X_{k+1} = X_k - {J(X_0)}^{-1} F(X_k)$, với $J(X) = \partial F(X) = \dfrac{\partial(f_1, f_2, \cdots)}{\partial(x_1, x_2, \cdots)}$

In [6]:
def modified_newton_method(initial_values, tol=1e-6, max_iter=100, *funcs):
    n = len(initial_values)  # Number of initial values (and functions)
    x = np.array(initial_values, dtype=float)  # Convert initial values to a NumPy array
    results = []  
    J_fixed = pd.DataFrame(funcs[n](*x))

    print("J(X_0) =")
    print(J_fixed.to_string(index=False, header=False))
    print("\nJ(X_0)^-1 =")
    J_inv = pd.DataFrame(np.linalg.inv(J_fixed.values))
    print(J_inv.to_string(index=False, header=False))
    print("-" * 50)

    for i in range(max_iter):
        F = np.array([func(*x) for func in funcs[:n]])
        
        delta_x = np.linalg.solve(J_fixed.values, -F)
        x_new = x + delta_x
        
        # Calculate the L1 norm (sum of absolute differences)
        norm_diff = np.sum(np.abs(x_new - x))
        results.append([i] + list(x_new) + [norm_diff])
        x = x_new
        
        if norm_diff < tol:
            break
    
    columns = ['k'] + [f'x({j+1}_k+1)' for j in range(n)] + ['norm_diff']
    df = pd.DataFrame(results, columns=columns)
    print(df.to_string(index=False))
    
    return x

In [ ]:
def f1(x, y):
    return 4*x**2 - 20*x + (1/4)*y**2 + 8

def f2(x, y):
    return (1/2)*x*y**2 + 2*x -5*y + 8

# Jacobian matrix function
def jacobian(x, y):
    return [
        [8*x-20, (1/2)*y],
        [(1/2)*y**2+2, x*y-5]
    ]

# Store functions and Jacobian function
funcs = [f1, f2, jacobian]

# Initial guess
initial_values = [0, 0]

# Perform Newton-Raphson method
solution = modified_newton_method(initial_values, 1e-6, 100, *funcs)
print("Approximate solution:", solution)